# Notebook 1 — Exploração de Dados e Visualização de Distribuições

## Aula 2: Detecção Estatística de Drift de Dados

### Objetivos
- Carregar e explorar o dataset sintético de crédito (fintech)
- Comparar visualmente distribuições de referência vs. produção
- Identificar quais features apresentam drift visual
- Compreender intuitivamente o deslocamento de distribuições

### Teoria-Chave (Documento 04)

> **Drift de dados** significa que a distribuição estatística dos dados de entrada mudou ao longo do tempo.
> "se antes a maioria dos tomadores de empréstimo tinha entre 30 e 50 anos, mas agora grande parte
> dos novos clientes é de jovens de 20 e poucos anos, a distribuição da variável 'idade' se deslocou."
>
> — Seção *Saiba Mais* do Documento da Aula 2

### Vídeo Relacionado
**Vídeo 1** — Detecção Estatística de Drift de Dados (15 min):  
Sinal de alerta em fintech; por que detectar drift proativamente; visão geral de métricas.

In [ ]:
# Imports
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Adiciona diretório raiz da aula ao path
sys.path.insert(0, str(Path.cwd().parent))

from src.data_preprocessing import DataPreprocessor

sns.set_theme(style="whitegrid")
%matplotlib inline

## 1. Geração / Carga do Dataset

O dataset simula dados de clientes de uma **fintech de crédito**, conforme o cenário
motivador do Documento da Aula 2. São 10.000 instâncias divididas em duas populações:

- **Referência (treinamento):** 5.000 clientes com perfil estável
- **Produção (com drift):** 5.000 clientes com perfil deslocado (mais jovens, menor renda)

In [ ]:
# Gera ou carrega o dataset
preprocessor = DataPreprocessor(seed=42, n_samples=5000)

data_path = Path.cwd().parent / "data" / "raw" / "dataset.csv"

if data_path.exists():
    ref_data, prod_data = preprocessor.load_data(str(data_path))
    print(f"Dataset carregado de: {data_path}")
else:
    print("Gerando dataset sintético...")
    df = preprocessor.generate_and_save(str(data_path))
    ref_data, prod_data = preprocessor.load_data(str(data_path))

print(f"Referência: {len(ref_data)} amostras")
print(f"Produção:   {len(prod_data)} amostras")

## 2. Visão Geral dos Dados

Inspecionamos a estrutura, tipos e estatísticas descritivas do dataset.

In [ ]:
# Informações gerais
full_data = pd.concat([ref_data, prod_data], ignore_index=True)
print(f"Shape total: {full_data.shape}")
print(f"\nColunas: {list(full_data.columns)}")
print(f"\nTipos:")
print(full_data.dtypes)
print(f"\nValores nulos: {full_data.isnull().sum().sum()}")

In [ ]:
# Estatísticas descritivas por grupo (referência vs produção)
print("=" * 60)
print("REFERÊNCIA (treinamento)")
print("=" * 60)
display(ref_data.describe().round(2))

print("\n" + "=" * 60)
print("PRODUÇÃO (com drift)")
print("=" * 60)
display(prod_data.describe().round(2))

## 3. Separação de Features Numéricas e Categóricas

Conforme o Documento da Aula 2, precisamos analisar drift tanto em **features contínuas**
(usando KS test, PSI) quanto em **features categóricas** (usando Qui-quadrado).

In [ ]:
# Identificar features numéricas e categóricas
_, numerical_cols, categorical_cols = preprocessor.prepare_features(ref_data)

print(f"Features numéricas ({len(numerical_cols)}): {numerical_cols}")
print(f"Features categóricas ({len(categorical_cols)}): {categorical_cols}")

## 4. Comparação Visual de Distribuições Numéricas

> "A forma mais intuitiva de começar é comparando diretamente os dados recentes com
> os dados históricos de referência. Colocar histogramas ou gráficos de densidade
> lado a lado ajuda a enxergar deslocamentos."
>
> — Seção *Saiba Mais — Como identificar mudanças de distribuição?*, Documento da Aula 2

Vamos comparar os histogramas de cada feature numérica entre referência e produção.

In [ ]:
# Histogramas comparativos para todas as features numéricas
fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.ravel()

for i, col in enumerate(numerical_cols):
    ax = axes[i]
    ax.hist(
        ref_data[col].dropna(), bins=40, alpha=0.5, density=True,
        label="Referência", color="steelblue", edgecolor="white",
    )
    ax.hist(
        prod_data[col].dropna(), bins=40, alpha=0.5, density=True,
        label="Produção", color="darkorange", edgecolor="white",
    )
    ax.set_title(col, fontsize=11, fontweight="bold")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

# Desativar eixos extras
for j in range(len(numerical_cols), len(axes)):
    axes[j].set_visible(False)

fig.suptitle(
    "Comparação de Distribuições: Referência vs. Produção\n"
    "(Conforme Figura 1 do Documento da Aula 2)",
    fontsize=14, fontweight="bold", y=1.02,
)
fig.tight_layout()
plt.show()

## 5. Gráficos de Densidade (KDE)

KDE plots oferecem uma visão mais suave das distribuições, facilitando a percepção
de deslocamentos na média e dispersão.

In [ ]:
# KDE plots para feature com maior drift visual: 'idade'
# Conforme o Documento: "clientes mais jovens na nova população"
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, col in zip(axes, ["idade", "renda_mensal", "score_credito"]):
    sns.kdeplot(ref_data[col], ax=ax, label="Referência", color="steelblue", fill=True, alpha=0.3)
    sns.kdeplot(prod_data[col], ax=ax, label="Produção", color="darkorange", fill=True, alpha=0.3)
    ax.set_title(f"Densidade: {col}", fontsize=12, fontweight="bold")
    ax.legend()
    ax.grid(True, alpha=0.3)

fig.suptitle("Features com Maior Drift Visual", fontsize=14, fontweight="bold")
fig.tight_layout()
plt.show()

## 6. Comparação de Distribuições Categóricas

Para variáveis categóricas como `tipo_residencia` e `categoria_risco`, analisamos
mudanças nas proporções. O teste Qui-quadrado (χ²) será aplicado no Notebook 2 para
verificar significância.

In [ ]:
# Comparação de distribuições categóricas
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col in zip(axes, categorical_cols):
    ref_counts = ref_data[col].value_counts(normalize=True).sort_index()
    prod_counts = prod_data[col].value_counts(normalize=True).sort_index()

    x = np.arange(len(ref_counts))
    width = 0.35

    ax.bar(x - width/2, ref_counts.values, width, label="Referência", color="steelblue")
    ax.bar(x + width/2, prod_counts.values, width, label="Produção", color="darkorange")
    ax.set_xticks(x)
    ax.set_xticklabels(ref_counts.index, rotation=45)
    ax.set_title(f"Proporções: {col}", fontsize=12, fontweight="bold")
    ax.set_ylabel("Proporção")
    ax.legend()
    ax.grid(True, alpha=0.3, axis="y")

fig.tight_layout()
plt.show()

## 7. Boxplots Comparativos

Boxplots ajudam a visualizar mudanças na mediana, dispersão e outliers.

In [ ]:
# Boxplots comparativos
fig, axes = plt.subplots(2, 4, figsize=(18, 10))
axes = axes.ravel()

for i, col in enumerate(numerical_cols):
    ax = axes[i]
    data_box = pd.DataFrame({
        "Valor": pd.concat([ref_data[col], prod_data[col]]),
        "Grupo": ["Referência"] * len(ref_data) + ["Produção"] * len(prod_data),
    })
    sns.boxplot(data=data_box, x="Grupo", y="Valor", ax=ax,
                palette={"Referência": "steelblue", "Produção": "darkorange"})
    ax.set_title(col, fontsize=11, fontweight="bold")
    ax.grid(True, alpha=0.3)

for j in range(len(numerical_cols), len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Boxplots: Referência vs. Produção", fontsize=14, fontweight="bold")
fig.tight_layout()
plt.show()

## 8. Correlação entre Features

Analisamos se a estrutura de correlação mudou entre referência e produção.

In [ ]:
# Heatmaps de correlação
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Referência
corr_ref = ref_data[numerical_cols].corr()
sns.heatmap(corr_ref, annot=True, fmt=".2f", cmap="coolwarm", ax=axes[0],
            vmin=-1, vmax=1, center=0, linewidths=0.5)
axes[0].set_title("Correlação — Referência", fontsize=12, fontweight="bold")

# Produção
corr_prod = prod_data[numerical_cols].corr()
sns.heatmap(corr_prod, annot=True, fmt=".2f", cmap="coolwarm", ax=axes[1],
            vmin=-1, vmax=1, center=0, linewidths=0.5)
axes[1].set_title("Correlação — Produção", fontsize=12, fontweight="bold")

fig.tight_layout()
plt.show()

## 9. Tabela Resumo de Diferenças

Comparamos as médias e desvios-padrão das features numéricas entre os dois grupos
para quantificar preliminarmente o deslocamento.

In [ ]:
# Tabela resumo de diferenças
summary = pd.DataFrame({
    "Ref (μ)": ref_data[numerical_cols].mean(),
    "Prod (μ)": prod_data[numerical_cols].mean(),
    "Δ μ": prod_data[numerical_cols].mean() - ref_data[numerical_cols].mean(),
    "Ref (σ)": ref_data[numerical_cols].std(),
    "Prod (σ)": prod_data[numerical_cols].std(),
    "Δ σ": prod_data[numerical_cols].std() - ref_data[numerical_cols].std(),
}).round(2)

print("Diferenças entre Referência e Produção:")
display(summary)

## 10. Distribuição do Target (Inadimplente)

Verificamos se a taxa de inadimplência também mudou — o que pode indicar
**label drift** (prior probability shift).

In [ ]:
# Comparação da taxa de inadimplência
fig, ax = plt.subplots(figsize=(8, 5))

labels = ["Referência", "Produção"]
rates = [
    ref_data["inadimplente"].mean(),
    prod_data["inadimplente"].mean(),
]

bars = ax.bar(labels, rates, color=["steelblue", "darkorange"], edgecolor="white")
for bar, rate in zip(bars, rates):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{rate:.1%}", ha="center", fontsize=12, fontweight="bold")

ax.set_ylabel("Taxa de Inadimplência")
ax.set_title("Taxa de Inadimplência: Referência vs. Produção", fontsize=13, fontweight="bold")
ax.set_ylim(0, max(rates) * 1.3)
ax.grid(True, alpha=0.3, axis="y")
fig.tight_layout()
plt.show()

## Resumo

Neste notebook, exploramos visualmente o dataset de crédito e identificamos evidências
claras de **data drift** em diversas features:

- **Idade:** deslocamento da média de ~40 para ~30 anos
- **Renda mensal:** redução de ~R$ 5.000 para ~R$ 3.500
- **Score de crédito:** redução e aumento de dispersão
- **Taxa de utilização de crédito:** aumento significativo
- **Tipo de residência e categoria de risco:** mudanças proporcionais

Conforme destacado no Documento da Aula 2:
> "A percepção visual é subjetiva e não quantifica formalmente o grau de diferença
> nem sua significância estatística."

No **Notebook 2**, aplicaremos **métricas formais** (KS, PSI, KL, JS) para quantificar
o drift e determinar se as diferenças são estatisticamente significativas.

---

**Próximo:** [02_treinamento.ipynb](02_treinamento.ipynb) — Implementação de KS, KL, JS e PSI